# 🐱 vs 🐶 Cats vs Dogs CNN — From Scratch (PyTorch)

**One notebook that runs on Google Colab, Kaggle, and your local machine.**

- ✅ Convolutional Neural Network built **from scratch** (no transfer learning)
- ✅ Real **Microsoft Dogs vs Cats** dataset (25k photos)
- ✅ Training curves, confusion matrix, Accuracy / Precision / Recall / F1
- ✅ Saves a trained model you can load anywhere
- ✅ Auto-detects Colab / Kaggle / local environments

**Runtime → Run all** on your platform of choice.

## 1 · Setup & environment detection

In [ ]:
# Install only what's missing for your platform
import importlib, subprocess, sys

def need(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                                pip_name or pkg])

for pkg in ["torch", "torchvision", "numpy", "matplotlib", "sklearn", "PIL"]:
    need(pkg)

import os, sys, math, random, time
import numpy as np
import matplotlib.pyplot as plt

import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from PIL import Image

# ---------------------------------------------------------------
# Auto-detect the platform
# ---------------------------------------------------------------
if os.path.exists("/kaggle"):
    PLATFORM = "kaggle"
elif os.path.exists("/content") and os.path.isdir("/content"):
    try:
        import google.colab  # type: ignore
        PLATFORM = "colab"
    except Exception:
        PLATFORM = "local"
else:
    PLATFORM = "local"

# device: use GPU whenever available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print(f"Platform : {PLATFORM}")
print(f"Device   : {device}")
print(f"PyTorch  : {torch.__version__}")

## 2 · Configuration

In [ ]:
# ---------- hyperparameters ----------
IMAGE_SIZE   = 128
IMAGES_PER_CLASS = 1200      # per class when downloading
TRAIN_SPLIT  = 0.80
EPOCHS       = 30
BATCH_SIZE   = 32
LR           = 1e-3
WEIGHT_DECAY = 1e-4
SEED         = 42
config_aug_rot = 5     # augmentation: gentle rotation (degrees)
config_aug_br  = 0.05  # augmentation: brightness jitter
config_aug_cn  = 0.05  # augmentation: contrast jitter

# model architecture (4 double-conv blocks + global average pooling)
CONV_DIMS = [32, 64, 128, 256]
FC, DROPOUT = 128, 0.5

# normalization is COMPUTED from the dataset below (mean/std of the pixel
# values). The checkpoint stores these so predictions always match training.

## 3 · Load the real dataset

- **Kaggle**: uses the built-in [Dogs vs Cats](https://www.kaggle.com/c/dogs-vs-cats) dataset at `/kaggle/input/dogs-vs-cats/`
- **Colab / local**: streams `N` images per class from Hugging Face (`microsoft/cats_vs_dogs`)

In [ ]:
from datasets import load_dataset as hf_load

def download_to(dirs, per_class, hf_id="microsoft/cats_vs_dogs"):
    """Stream real photos from Hugging Face into cats/ and dogs/ folders."""
    os.makedirs(dirs["cat"], exist_ok=True)
    os.makedirs(dirs["dog"], exist_ok=True)
    counts = {"cat": 0, "dog": 0}
    ds = hf_load(hf_id, split="train", streaming=True)
    for row in ds:
        img = row.get("image")
        if img is None:
            continue
        label = row.get("labels", row.get("label"))
        name = label.lower() if isinstance(label, str) else ("dog" if int(label) == 1 else "cat")
        if counts[name] >= per_class:
            if counts["cat"] >= per_class and counts["dog"] >= per_class:
                break
            continue
        out = os.path.join(dirs[name], f"{name}_{counts[name]+1}.jpg")
        img.convert("RGB").save(out, "JPEG")
        counts[name] += 1
        if counts["cat"] >= per_class and counts["dog"] >= per_class:
            break
    print("Downloaded:", counts)
    return counts

if PLATFORM == "kaggle":
    # Kaggle ships the full competition dataset in the input folder
    src = "/kaggle/input/dogs-vs-cats/train"
    cats, dogs = [], []
    for f in sorted(os.listdir(src)):
        p = os.path.join(src, f)
        (cats if f.startswith("cat.") else dogs).append(p)
    print(f"Kaggle dataset: {len(cats)} cats / {len(dogs)} dogs found")
    cat_paths, dog_paths = cats[:IMAGES_PER_CLASS], dogs[:IMAGES_PER_CLASS]
else:
    # Download a slice of the real dataset (disk friendly)
    dirs = {"cat": "data/cats", "dog": "data/dogs"}
    if not (os.path.isdir(dirs["cat"]) and os.listdir(dirs["cat"])):
        download_to(dirs, IMAGES_PER_CLASS)
    cat_paths = sorted(os.path.join(dirs["cat"], f) for f in os.listdir(dirs["cat"]))
    dog_paths = sorted(os.path.join(dirs["dog"], f) for f in os.listdir(dirs["dog"]))

print(f"Using {len(cat_paths)} cats + {len(dog_paths)} dogs")

In [ ]:
def load_images(paths, label, image_size=IMAGE_SIZE):
    """Return (uint8 HWC array, labels)."""
    arr, labels = [], []
    for p in paths:
        try:
            img = Image.open(p).convert("RGB").resize((image_size, image_size))
            arr.append(np.asarray(img, dtype=np.uint8)); labels.append(label)
        except Exception as e:
            print("[skip]", p, e)
    return np.array(arr), np.array(labels)

Xc, yc = load_images(cat_paths, 0)
Xd, yd = load_images(dog_paths, 1)
X = np.concatenate([Xc, Xd]); y = np.concatenate([yc, yd])
print("Dataset:", X.shape, "labels:", y.shape, "| cats:", int((y==0).sum()), "dogs:", int((y==1).sum()))

# dataset-specific normalization from the TRAIN split only (no val leakage) -
# stored with the checkpoint and used for ALL preprocessing.
import torch
MEAN = [float(Xtr[..., c].mean() / 255.0) for c in range(3)]
STD  = [float(Xtr[..., c].std()  / 255.0) for c in range(3)]
print("Pixel mean:", [round(m, 4) for m in MEAN])
print("Pixel std :", [round(s, 4) for s in STD])
NORMALIZE = transforms.Normalize(MEAN, STD)

# leak-free stratified split (before augmentation!)
from sklearn.model_selection import train_test_split
Xtr, Xva, ytr, yva = train_test_split(
    X, y, test_size=1-TRAIN_SPLIT, random_state=SEED, stratify=y, shuffle=True)
print(f"Train {len(Xtr)} / Val {len(Xva)}")

In [ ]:
# preview a few real photos
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, (img, lab) in zip(axes.flat, list(zip(X[:6], y[:6])) + list(zip(X[-6:], y[-6:]))):
    ax.imshow(img); ax.axis("off")
    ax.set_title("cat" if lab == 0 else "dog")
plt.suptitle("Real Dogs-vs-Cats samples", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

## 4 · The CNN (built from scratch)

In [ ]:
class CatDogCNN(nn.Module):
    """From-scratch CNN: 4 double-conv blocks (BN + ReLU + MaxPool),
       global average pooling, then 2 dense layers.
       No pretrained weights / transfer learning anywhere.
       Key names match src/model.py so weights load 1:1."""

    def __init__(self, image_size=128):
        super().__init__()
        blocks, in_ch = [], 3
        for out_ch in CONV_DIMS:
            blocks.append(nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            ))
            in_ch = out_ch
        self.features = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(DROPOUT),
            nn.Linear(in_ch, FC), nn.ReLU(inplace=True), nn.Dropout(DROPOUT),
            nn.Linear(FC, 1),
        )

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

model = CatDogCNN(IMAGE_SIZE).to(device)
print(model)
print(f"Trainable parameters: {model.num_params():,}")

## 5 · Transforms, augmentation & DataLoaders

In [ ]:
def to_tensor(arr):
    t = torch.from_numpy(arr).permute(0, 3, 1, 2).float() / 255.0
    return NORMALIZE(t)

Xtr_t, Xva_t = to_tensor(Xtr), to_tensor(Xva)
ytr_t, yva_t = torch.from_numpy(ytr).long(), torch.from_numpy(yva).long()

# Real-time augmentation (training only, in-pipeline)
aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=config_aug_rot),
    transforms.ColorJitter(brightness=config_aug_br, contrast=config_aug_cn),
])

class AugDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, augment=False):
        self.X, self.y, self.augment = X, y, augment
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x, y = self.X[i], self.y[i]
        if self.augment:
            x = aug(x)
        return x, y

train_ds = AugDataset(Xtr_t, ytr_t, augment=True)
val_ds   = AugDataset(Xva_t, yva_t, augment=False)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

print(f"train batches: {len(train_dl)} | val batches: {len(val_dl)}")

## 6 · Train

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
# cosine annealing from LR down to 1e-5 across all epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc, best_state = 0.0, None
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    # --- train ---
    model.train(); tl = tc = n = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb).squeeze(1)
        loss = criterion(logits, yb.float())
        loss.backward(); optimizer.step()
        tl += loss.item() * xb.size(0)
        tc += ((torch.sigmoid(logits) >= 0.5).long() == yb).sum().item()
        n += xb.size(0)
    tloss, tacc = tl / n, tc / n

    # --- validate ---
    model.eval(); vl = vc = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb).squeeze(1)
            vl += criterion(logits, yb.float()).item() * xb.size(0)
            vc += ((torch.sigmoid(logits) >= 0.5).long() == yb).sum().item()
    vloss, vacc = vl / len(val_ds), vc / len(val_ds)

    history["train_loss"].append(tloss); history["train_acc"].append(tacc)
    history["val_loss"].append(vloss);   history["val_acc"].append(vacc)

    if vacc > best_val_acc:
        best_val_acc = vacc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    scheduler.step()   # cosine annealing

    print(f"Epoch {epoch:>2}/{EPOCHS} | loss={tloss:.4f} acc={tacc:.4f} | "
          f"val_loss={vloss:.4f} val_acc={vacc:.4f}")

if best_state:
    model.load_state_dict(best_state)
print(f"Done in {time.time()-t0:.1f}s | best val acc = {best_val_acc:.4f}")

## 7 · Evaluation: metrics & plots

In [ ]:
# restore best weights for evaluation
if best_state: model.load_state_dict(best_state)
model.eval()

probs, labels = [], []
with torch.no_grad():
    for xb, yb in val_dl:
        probs.append(torch.sigmoid(model(xb.to(device)).squeeze(1)).cpu().numpy())
        labels.append(yb.numpy())
probs = np.concatenate(probs); labels = np.concatenate(labels)
preds = (probs >= 0.5).astype(int)

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)

acc  = accuracy_score(labels, preds)
prec = precision_score(labels, preds, zero_division=0)
rec  = recall_score(labels, preds, zero_division=0)
f1   = f1_score(labels, preds, zero_division=0)
cm   = confusion_matrix(labels, preds)

print("=" * 58)
print(" MODEL EVALUATION METRICS ON VALIDATION SET")
print("=" * 58)
print(f"  * Accuracy        : {acc*100:.2f}%")
print(f"  * Precision       : {prec*100:.2f}%")
print(f"  * Recall (Sens.)  : {rec*100:.2f}%")
print(f"  * F1-Score        : {f1*100:.2f}%")
print("=" * 58)

In [ ]:
# training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history["train_acc"], label="Training Accuracy", color="#1f77b4", lw=2.5)
ax1.plot(history["val_acc"], label="Validation Accuracy", color="#ff7f0e", ls="--", lw=2.5)
ax1.set(title="Accuracy vs Epochs", xlabel="Epoch", ylabel="Accuracy")
ax1.legend(); ax1.grid(True, ls=":", alpha=.6)

ax2.plot(history["train_loss"], label="Training Loss", color="#1f77b4", lw=2.5)
ax2.plot(history["val_loss"], label="Validation Loss", color="#ff7f0e", ls="--", lw=2.5)
ax2.set(title="Loss vs Epochs", xlabel="Epoch", ylabel="BCE Loss")
ax2.legend(); ax2.grid(True, ls=":", alpha=.6)
plt.tight_layout()
plt.savefig("training_history.png", dpi=200, bbox_inches="tight")
plt.show()

# confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=["Cat", "Dog"]).plot(cmap=plt.cm.Blues, ax=ax, values_format="d")
ax.set_title("Confusion Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

## 8 · Save & export the trained model

In [ ]:
# checkpoint: weights + config + labels + normalization in one file so
# the saved model is fully self-describing (src/predict.py reads mean/std)
ckpt = {
    "state_dict": model.state_dict(),
    "image_size": IMAGE_SIZE,
    "labels": ["cat", "dog"],
    "mean": MEAN,
    "std": STD,
    "metrics": {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1},
}
os.makedirs("models", exist_ok=True)
torch.save(ckpt, "models/cat_dog_cnn.pt")
print("Saved models/cat_dog_cnn.pt")

if PLATFORM == "kaggle":
    # available under /kaggle/working/ for Download as zip
    torch.save(ckpt, "/kaggle/working/cat_dog_cnn.pt")
    print("Saved to /kaggle/working/cat_dog_cnn.pt")
elif PLATFORM == "colab":
    from google.colab import files
    files.download("models/cat_dog_cnn.pt")   # triggers browser download

## 9 · Predict on an image

In [ ]:
def predict_img_from_array(arr, model=model):
    """Predict from an HWC uint8 array."""
    x = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    x = (x.float() / 255.0 - torch.tensor(MEAN).view(1,3,1,1)) / torch.tensor(STD).view(1,3,1,1)
    with torch.no_grad():
        p = torch.sigmoid(model(x.to(device))).item()
    label = "dog" if p >= 0.5 else "cat"
    conf = p*100 if label == "dog" else (1-p)*100
    return label, conf

In [ ]:
def predict_img(path, model=model, image_size=IMAGE_SIZE):
    img = Image.open(path).convert("RGB").resize((image_size, image_size))
    x = torch.from_numpy(np.asarray(img, dtype=np.uint8)).permute(2, 0, 1).unsqueeze(0)
    x = (x.float() / 255.0 - torch.tensor(MEAN).view(1,3,1,1)) / torch.tensor(STD).view(1,3,1,1)
    with torch.no_grad():
        p = torch.sigmoid(model(x.to(device))).item()
    label = "dog" if p >= 0.5 else "cat"
    conf = p*100 if label == "dog" else (1-p)*100
    return label, conf

# demo on 4 random validation images
idx = np.random.choice(len(Xva), 4, replace=False)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, i in zip(axes, idx):
    lbl, cf = predict_img_from_array(Xva[i])
    ax.imshow(Xva[i]); ax.axis("off")
    truth = "cat" if yva[i] == 0 else "dog"
    color = "green" if lbl == truth else "red"
    ax.set_title(f"Pred: {lbl} ({cf:.1f}%)\nTrue: {truth}", color=color, fontsize=11)
plt.suptitle("Sample Predictions", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

## ✅ Done

**Next steps**

| Platform | What's next |
|----------|-------------|
| **Local** | `uvicorn api.main:app` → open http://localhost:8000 for the upload UI |
| **Colab** | model auto-downloaded to your machine (`.pt`) |
| **Kaggle** | model in `/kaggle/working/cat_dog_cnn.pt` → Download as zip |
| **Host** | See `README.md` (Render / Railway / Docker) |